In [1]:
%%capture
import pandas as pd
from pathlib import Path
import plotly.express as px

import torch
from nnsight import LanguageModel
from transformers import AutoTokenizer

dir = Path('../data/transcripts')
doc_paths = sorted(dir.glob('*.txt'))

# tokenize + collect activations
# id = 'allenai/Olmo-3-1125-32B'
id = 'Qwen/Qwen3-30B-A3B-Instruct-2507'
model = LanguageModel(id, device_map = 'auto', remote = 'true')
tokenizer = AutoTokenizer.from_pretrained(id)

In [2]:
with open(doc_paths[3], "r", encoding="utf-8") as f:
    michael = tokenizer.apply_chat_template([{'role': 'user', 'content': 'Think about this conversation and try your best to distinguish the people who are involved' + f.read()}], tokenize = False)

with open(doc_paths[4], "r", encoding="utf-8") as f:
    mike = tokenizer.apply_chat_template([{'role': 'user', 'content': 'Think about this conversation and try your best to distinguish the people who are involved' + f.read()}], tokenize = False)

In [3]:
# get tokens
michael_toks = tokenizer(michael, return_offsets_mapping = True)
mike_toks = tokenizer(mike, return_offsets_mapping = True)

def get_token_map(token_dict, string):
    token_map = {}
    token_map['token_id'] = token_dict['input_ids']
    token_map['offsets'] = token_dict['offset_mapping']
    token_map['token'] = [string[o[0]:o[1]] for o in token_map['offsets']]
    token_map['token_ix'] = [i for i in range(len(token_map['token']))]
    return pd.DataFrame(token_map)

michael_map = get_token_map(michael_toks, michael)
mike_map = get_token_map(mike_toks, mike)


In [4]:
# get hidden states
michael_hs = None
with model.trace(michael, remote = True):
    michael_hs = model.model.layers[24].output[0].save()

mike_hs = None
with model.trace(mike, remote = True):
    mike_hs = model.model.layers[24].output[0].save()


⬇ Downloading: 100%|██████████| 1.24M/1.24M [00:01<00:00]


⬇ Downloading: 100%|██████████| 1.24M/1.24M [00:00<00:00]


In [5]:
# now, we want a way to indicate which segments are "where the experiment is happening"
from utils.utils import flag_message_types
michael_map_labeled = flag_message_types(michael_map, base_messages = ['oh, well have fun at tennis, I think later we will go to the Fresh Market and get some pasta and salmon, goodbye!'])
mike_map_labeled = flag_message_types(mike_map, base_messages = ['oh, well have fun at tennis, I think later we will go to the Fresh Market and get some pasta and salmon, goodbye!'])


In [19]:
import numpy as np
import torch
from sklearn.decomposition import PCA

skip_n = 11

# Slice token maps to post-skip_n portion (aligns with hs tensor indices)
michael_map_slice = michael_map_labeled.iloc[skip_n:].reset_index(drop=True)
mike_map_slice = mike_map_labeled.iloc[skip_n:].reset_index(drop=True)

# Filter to experimental segment only
michael_exp_mask = michael_map_slice['base_message_ix'] == 0.0
mike_exp_mask = mike_map_slice['base_message_ix'] == 0.0

michael_exp_pos = michael_exp_mask[michael_exp_mask].index.tolist()
mike_exp_pos = mike_exp_mask[mike_exp_mask].index.tolist()

# Extract hidden states only for experimental segment tokens
michael_exp_hs = michael_hs[skip_n:][michael_exp_pos].float().numpy()
mike_exp_hs = mike_hs[skip_n:][mike_exp_pos].float().numpy()

# Fit PCA only on these points
comb = np.concatenate([michael_exp_hs, mike_exp_hs], axis=0)
pca = PCA(n_components=2)
coords = pca.fit_transform(comb)

michael_hs_pca = pd.DataFrame(coords[:len(michael_exp_pos), :])
mike_hs_pca = pd.DataFrame(coords[len(michael_exp_pos):, :])

print(michael_hs_pca, mike_hs_pca)

            0         1
0    4.665953 -4.004670
1    4.044372 -1.546472
2    6.098412 -2.090892
3    4.021535 -2.400788
4    1.555913 -3.823715
5    0.563817 -1.185095
6   -3.379234 -3.389205
7    2.682259 -2.018109
8    2.999419  0.511707
9    3.815925  3.044647
10   2.778305  3.562331
11   2.451261  2.452471
12   3.461971  4.568954
13   1.645046  4.831495
14  -0.010637  4.230793
15  -0.910769  0.674072
16  -4.294864 -3.370182
17  -7.214102 -0.167760
18  -0.276166  3.581419
19  -3.351828  3.294108
20  -4.209787  2.418813
21 -10.225744 -0.593442
22  -3.168654  2.713836
23  -9.474719 -2.342371
24   0.269284  1.026402
25   1.990278 -6.715414
26   2.549200 -3.264328             0         1
0    4.668898 -4.254269
1    3.726718 -1.351555
2    5.825665 -2.175491
3    3.749095 -2.433062
4    1.333986 -4.291587
5    0.439419 -1.547386
6   -3.357599 -3.291466
7    2.855471 -1.931874
8    2.993750  0.662518
9    3.981832  3.025160
10   2.947368  3.595335
11   2.406956  2.700993
12   3.554105  4

In [20]:
# plotting — only experimental segment tokens
michael_plot = pd.concat([michael_map_slice[michael_exp_mask].reset_index(drop=True), michael_hs_pca], axis=1)
michael_plot['is_sequence'] = 'Michael'

mike_plot = pd.concat([mike_map_slice[mike_exp_mask].reset_index(drop=True), mike_hs_pca], axis=1)
mike_plot['is_sequence'] = 'Mike'

In [22]:
michael_plot

,token_id,offsets,token,token_ix,base_message_ix,base_message,0,1,is_sequence
0,14019,"(1262, 1265)",oh,356,0.0,"oh, well have fun at tennis, I think later we ...",4.665953,-4.004670,Michael
1,11,"(1265, 1266)",",",357,0.0,"oh, well have fun at tennis, I think later we ...",4.044372,-1.546472,Michael
2,1632,"(1266, 1271)",well,358,0.0,"oh, well have fun at tennis, I think later we ...",6.098412,-2.090892,Michael
3,614,"(1271, 1276)",have,359,0.0,"oh, well have fun at tennis, I think later we ...",4.021535,-2.400788,Michael
4,2464,"(1276, 1280)",fun,360,0.0,"oh, well have fun at tennis, I think later we ...",1.555913,-3.823715,Michael
5,518,"(1280, 1283)",at,361,0.0,"oh, well have fun at tennis, I think later we ...",0.563817,-1.185095,Michael
6,31415,"(1283, 1290)",tennis,362,0.0,"oh, well have fun at tennis, I think later we ...",-3.379234,-3.389205,Michael
7,11,"(1290, 1291)",",",363,0.0,"oh, well have fun at tennis, I think later we ...",2.682259,-2.018109,Michael
8,358,"(1291, 1293)",I,364,0.0,"oh, well have fun at tennis, I think later we ...",2.999419,0.511707,Michael
9,1744,"(1293, 1299)",think,365,0.0,"oh, well have fun at tennis, I think later we ...",3.815925,3.044647,Michael


In [23]:
plot_df = pd.concat([michael_plot, mike_plot], axis=0)
plot_df['color'] = plot_df['is_sequence']

In [24]:
import plotly.express as px
plot = px.scatter(plot_df, x = plot_df[0], y = plot_df[1], color = 'color', hover_data = ['token'])
plot

In [17]:
new_plot = px.scatter(plot_df, x = plot_df[0], y = plot_df[1], color = 'color', hover_data = ['token'])
new_plot

In [210]:
# this doesn't show much, but we expect the variance to be pretty small because it'll emphasize the 
# directions of highest variance (which could result from something else :))
plot

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'customdata': array([['Michael'],
                                   [':'],
                                   [' Hey'],
                                   ...,
                                   ['.\n'],
                                   ['Mike'],
                                   [':']], shape=(590, 1), dtype=object),
              'hovertemplate': 'color=Other<br>0=%{x}<br>1=%{y}<br>token=%{customdata[0]}<extra></extra>',
              'legendgroup': 'Other',
              'marker': {'color': '#636efa', 'symbol': 'circle'},
              'mode': 'markers',
              'name': 'Other',
              'orientation': 'v',
              'showlegend': True,
              'type': 'scatter',
              'x': {'bdata': ('50kdwSNnNMHxm5/BPViKwYRZxME9BK' ... 'E/+RaiQNO9HEGstr7BeBXRwVtsqMA='),
                    'dtype': 'f4'},
              'xaxis': 'x',
              'y': {'bdata': ('pooCwfMRLcAtgHtAHQjyQA7Gvz7Tdq' ... '5A3Pmtv0Epo0D0vYLBJlp3wcN5x0A='),
                    'dtype': 'f4'},
              'yaxis': 'y'},
             {'customdata': array([[' oh'],
                                   [','],
                                   [' well'],
                                   [' have'],
                                   [' fun'],
                                   [' at'],
                                   [' tennis'],
                                   [','],
                                   [' goodbye'],
                                   ['!']], dtype=object),
              'hovertemplate': 'color=Michael<br>0=%{x}<br>1=%{y}<br>token=%{customdata[0]}<extra></extra>',
              'legendgroup': 'Michael',
              'marker': {'color': '#EF553B', 'symbol': 'circle'},
              'mode': 'markers',
              'name': 'Michael',
              'orientation': 'v',
              'showlegend': True,
              'type': 'scatter',
              'x': {'bdata': '94q3wCgwvsDotHTArf1swEHarz/x4RpA1QZIQJI4xL9oddTAYyFdwQ==', 'dtype': 'f4'},
              'xaxis': 'x',
              'y': {'bdata': 'rRnxQKBq+EDI1ARBetoqQQN1Y0GlTJdAC0HxQPoAJkE2kD5BgWkxPw==', 'dtype': 'f4'},
              'yaxis': 'y'},
             {'customdata': array([[' oh'],
                                   [','],
                                   [' well'],
                                   [' have'],
                                   [' fun'],
                                   [' at'],
                                   [' tennis'],
                                   [','],
                                   [' goodbye'],
                                   ['!']], dtype=object),
              'hovertemplate': 'color=Mike<br>0=%{x}<br>1=%{y}<br>token=%{customdata[0]}<extra></extra>',
              'legendgroup': 'Mike',
              'marker': {'color': '#00cc96', 'symbol': 'circle'},
              'mode': 'markers',
              'name': 'Mike',
              'orientation': 'v',
              'showlegend': True,
              'type': 'scatter',
              'x': {'bdata': 'm3CJwHlA7r8CYjbA04TKv7eHoT7DGJ++eW4YQCKjScAjadfA/HhHwQ==', 'dtype': 'f4'},
              'xaxis': 'x',
              'y': {'bdata': 'R0nUQMP5rUAa/MxAG6v2QDIscEHzTQJBNUQKQc1dMUGNTERBxnRXPw==', 'dtype': 'f4'},
              'yaxis': 'y'}],
    'layout': {'legend': {'title': {'text': 'color'}, 'tracegroupgap': 0},
               'margin': {'t': 60},
               'template': '...',
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': '0'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': '1'}}}
})